# SageMaker V3 JumpStart E2E Training Example

This notebook demonstrates how to use SageMaker V3 to train a JumpStart model from scratch and deploy it for inference.

### Prerequisites
Note: Ensure you have sagemaker and ipywidgets installed in your environment. The ipywidgets package is required to monitor endpoint deployment progress in Jupyter notebooks.


In [1]:
# Import required libraries
import json
import uuid

from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.core.resources import EndpointConfig

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Step 1: Configure JumpStart Model for Training

We'll train a HuggingFace Falcon model using JumpStart.

In [2]:
# Configuration
MODEL_ID = "huggingface-spc-bert-base-cased"
MODEL_NAME_PREFIX = "js-e2e-example-model"
ENDPOINT_NAME_PREFIX = "js-e2e-example-endpoint"

# Generate unique identifiers
unique_id = str(uuid.uuid4())[:8]
training_job_name = f"js-training-{unique_id}"
model_name = f"{MODEL_NAME_PREFIX}-{unique_id}"
endpoint_name = f"{ENDPOINT_NAME_PREFIX}-{unique_id}"

print(f"Training job name: {training_job_name}")
print(f"Model name: {model_name}")
print(f"Endpoint name: {endpoint_name}")

Training job name: js-training-285f9899
Model name: js-e2e-example-model-285f9899
Endpoint name: js-e2e-example-endpoint-285f9899


## Step 2: Train the Model

Use ModelTrainer to train the JumpStart model. The training job may take 30+ minutes to complete.

In [3]:
# Create JumpStart configuration
jumpstart_config = JumpStartConfig(model_id=MODEL_ID)

# Initialize ModelTrainer from JumpStart config
model_trainer = ModelTrainer.from_jumpstart_config(
    jumpstart_config=jumpstart_config, 
    base_job_name=training_job_name, 
    hyperparameters={"epochs": 1}
)

# Train the model
print("Starting model training...")
model_trainer.train()
print(f"Training completed: {training_job_name}")

[02/11/26 09:46:00] INFO     SageMaker session not provided. Using default Session.                  ]8;id=585928;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=524060;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Role not provided. Using default role:                                  ]8;id=457853;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=808478;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#75\75]8;;\
                             arn:aws:iam::975049911976:role/from-idea-to-prod-SageMakerExecutionRole               
                             -h5rrBhGSjbNO                                                                         

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=369742;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=53514;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#354\354]8;;\

                    INFO     hub_content_name: huggingface-spc-bert-base-cased, hub_content_version: ]8;id=140474;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/document.py\document.py]8;;\:]8;id=188203;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/document.py#71\71]8;;\
                             3.0.10                                                                                

                    INFO     Compute not provided. Using default compute:                           ]8;id=59525;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=659209;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#227\227]8;;\
                             instance_type='ml.p3.2xlarge' instance_count=1 volume_size_in_gb=30                   
                             volume_kms_key_id=None keep_alive_period_in_seconds=None                              
                             capacity_reservation_ids=None instance_groups=None                                    
                             capacity_schedules_config=None training_plan_arn=None                                 
                             instance_placement_config=None enable_managed_spot_training=None                      

...

                    INFO     js-training-285f9899-20260211094600/algo-1-1770803236:              ]8;id=916956;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=187810;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             Info file not found at                                                                
                             '_input_model_extracted/__models_info__.json'.                                        

                    INFO     js-training-285f9899-20260211094600/algo-1-1770803236:              ]8;id=230474;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=15560;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

                    INFO     js-training-285f9899-20260211094600/algo-1-1770803236:              ]8;id=876889;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=313807;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35869\35869]8;;\
                             Training Container Execution Completed                                                

[02/11/26 10:14:59] INFO     Final Resource Status: Completed                                    ]8;id=481323;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=691807;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#35872\35872]8;;\

Training completed: js-training-285f9899


## Step 3: Build ModelBuilder from Trained Model

Create a ModelBuilder from the training artifacts.

In [4]:
# Build ModelBuilder from trained model
model_builder = ModelBuilder(
    model=model_trainer,
    dependencies={"auto": False}
)

[02/11/26 10:15:00] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=871572;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=166574;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#337\337]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=761924;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=838199;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#369\369]8;;\

## Step 4: Build the Model

Build the model artifacts and prepare for deployment.

In [5]:
# Build the model
core_model = model_builder.build(model_name=model_name)
print(f"Model Successfully Created: {core_model.model_name}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=841623;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=193614;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    DEBUG    Either inference spec or model is provided. ModelBuilder   ]8;id=788039;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=783560;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1315\1315]8;;\
                             is not handling MLflow model input                                                    

Using model 'huggingface-spc-bert-base-cased' with wildcard version identifier '*'. You can pin to version '3.0.10' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


                    WARNING  Using model 'huggingface-spc-bert-base-cased' with wildcard version       ]8;id=443542;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/cache.py\cache.py]8;;\:]8;id=160559;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/jumpstart/cache.py#624\624]8;;\
                             identifier '*'. You can pin to version '3.0.10' for more stable results.              
                             Note that models may have different input/output signatures after a major             
                             version upgrade.                                                                      

                    DEBUG    JumpStart Model ID detected.                               ]8;id=168433;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=736386;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#2861\2861]8;;\

                    DEBUG    Building for JumpStart model ID...                               ]8;id=192477;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=504205;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#1800\1800]8;;\

                    WARNING  Unable to check docker volume utilization at the expected path   ]8;id=217152;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/utils/local_hardware.py\local_hardware.py]8;;\:]8;id=724601;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/utils/local_hardware.py#204\204]8;;\
                             /var/lib/docker. [Errno 2] No such file or directory:                                 
                             '/var/lib/docker'                                                                     

                    INFO     Creating model with name: js-e2e-example-model-285f9899         ]8;id=105236;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=130652;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1814\1814]8;;\

[02/11/26 10:15:01] INFO     ✅ Model has been created: 'js-e2e-example-model-285f9899' using ]8;id=742015;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=322725;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#2605\2605]8;;\
                             server MMS in SAGEMAKER_ENDPOINT mode (ARN:                                           
                             arn:aws:sagemaker:us-west-2:975049911976:model/js-e2e-example-mo                      
                             del-285f9899)                                                                         

Model Successfully Created: js-e2e-example-model-285f9899


## Step 5: Deploy the Trained Model

Deploy the trained model to a SageMaker endpoint for real-time inference.

In [6]:
# Deploy the trained model to an endpoint
core_endpoint = model_builder.deploy(endpoint_name=endpoint_name)
print(f"Endpoint Successfully Created: {core_endpoint.endpoint_name}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=12795;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=777690;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#92\92]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating endpoint-config with name                               ]8;id=127569;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=244977;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#985\985]8;;\
                             js-e2e-example-endpoint-285f9899                                                      

[02/11/26 10:15:02] INFO     Creating endpoint with name js-e2e-example-endpoint-285f9899    ]8;id=571439;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=930734;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#1017\1017]8;;\

Output()

Warning: MMS is using non-default JVM parameters: -XX:-UseContainerSupport

...

2026-02-11T10:18:16,373 [INFO ] pool-2-thread-3 ACCESS_LOG - /169.254.178.2:55304 "GET /ping HTTP/1.1" 200 0

2026-02-11T10:18:21,373 [INFO ] pool-2-thread-3 ACCESS_LOG - /169.254.178.2:55304 "GET /ping HTTP/1.1" 200 0

2026-02-11T10:18:26,373 [INFO ] pool-2-thread-3 ACCESS_LOG - /169.254.178.2:55304 "GET /ping HTTP/1.1" 200 0

[02/11/26 10:18:34] INFO     ✅ Deployment successful: Endpoint                               ]8;id=73768;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=909476;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#1968\1968]8;;\
                             'js-e2e-example-endpoint-285f9899' using MMS in                                       
                             SAGEMAKER_ENDPOINT mode (ARN:                                                         
                             arn:aws:sagemaker:us-west-2:975049911976:endpoint/js-e2e-example                      
                             -endpoint-285f9899)                                                                   

Endpoint Successfully Created: js-e2e-example-endpoint-285f9899


## Step 6: Test the Endpoint

This endpoint performs text entailment classification, determining the logical relationship between pairs of sentences. The returned scores indicate the model's confidence for different entailment categories (e.g., entailment, contradiction, neutral) - higher scores indicate stronger predictions for each relationship type. 

This sends a test request to the deployed endpoint.

In [7]:
# Test with a sample query for the trained model
test_data = ["The weather is sunny today", "It is not raining"]

result = core_endpoint.invoke(
    body=json.dumps(test_data),
    content_type="application/list-text"
)

entailment_scores = json.loads(result.body.read().decode('utf-8'))
print(f"Result of invoking trained endpoint: {entailment_scores}")


Result of invoking trained endpoint: {'probabilities': [1.2199833393096924, -1.2695157527923584]}


## Step 7: Clean Up Resources

Clean up the created resources to avoid ongoing charges.

In [8]:
# Clean up resources
core_endpoint_config = EndpointConfig.get(endpoint_config_name=core_endpoint.endpoint_name)

# Delete in the correct order
core_model.delete()
core_endpoint.delete()
core_endpoint_config.delete()

print("Trained Model and Endpoint Successfully Deleted!")

[02/11/26 10:18:35] INFO     Deleting Model - js-e2e-example-model-285f9899                      ]8;id=904870;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=392388;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#22918\22918]8;;\

                    INFO     Deleting Endpoint - js-e2e-example-endpoint-285f9899                ]8;id=849342;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=634043;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#10939\10939]8;;\

                    INFO     Deleting EndpointConfig - js-e2e-example-endpoint-285f9899          ]8;id=923344;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=699297;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11753\11753]8;;\

Trained Model and Endpoint Successfully Deleted!


## Summary

This notebook demonstrated:
1. Training a JumpStart model using ModelTrainer
2. Building a ModelBuilder from training artifacts
3. Building the model for deployment
4. Deploying to a SageMaker endpoint
5. Making inference requests
6. Cleaning up resources

The V3 ModelTrainer and ModelBuilder provide a seamless end-to-end workflow from training to deployment!